# 1. B-0 복원과 B30 pruning 준비

세션이 끊긴 뒤 새 GPU 런타임에서도 위에서부터 실행한다. B-0를 재학습하지 않고 Drive의 최고 체크포인트를 복원한다. 기존 코드와 데이터는 재사용하고 없으면 내려받는다.

B-0를 검증한 뒤 각 블록 MLP 내부 채널의 30%를 제거한다. 전체 parameter의 30%를 의미하지 않는다. 원본 B-0 파일은 보존한다.

`B30_from_B0_init.pth`가 이미 저장돼 있다면 이 노트북을 건너뛰고 회복 학습 노트북으로 이동한다.


## 1. Drive와 경로
기존 B-0 최고 모델 경로를 확인한다.


In [1]:
from google.colab import drive
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

import hashlib
import importlib.util
import json
import random
import subprocess
import sys
import time

import numpy as np
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

drive.mount("/content/drive")
assert torch.cuda.is_available(), "Colab 런타임 유형에서 GPU를 선택하세요."
device = torch.device("cuda")

b0_path = Path("/content/drive/MyDrive/ondeviceAI/checkpoints/B0_20261024_172759_400529/best.pth")
recovery_init_path = Path("/content/drive/MyDrive/ondeviceAI/initial_checkpoints/B30_from_B0_init.pth")
repo_dir = Path("/content/ondeviceAI")
data_dir = repo_dir / "data/CUB_200_2011/CUB_200_2011"

repo_url = "https://github.com/nicewei77-lang/ondeviceAI.git"
DATA_REF = "dev"

config = {
    "seed": 0,
    "epochs": 20,
    "batch_size": 8,
    "optimizer": "AdamW",
    "backbone_lr": 1e-5,
    "head_lr": 1e-4,
    "weight_decay": 0.01,
    "scheduler": None,
    "train_precision": "AMP float16",
    "validation_precision": "float32",
    "selection": "validation accuracy, then validation loss",
}

assert b0_path.is_file(), f"B-0 경로를 확인하세요: {b0_path}"
assert not recovery_init_path.exists(), (
    "B30_from_B0_init.pth가 이미 있습니다. 준비 노트북을 건너뛰고 회복 학습 노트북을 실행하세요."
)
print("GPU:", torch.cuda.get_device_name(0))
print("B-0 원본 파일:", b0_path)


Mounted at /content/drive
GPU: Tesla T4
B-0 원본 파일: /content/drive/MyDrive/ondeviceAI/checkpoints/B0_20261024_172759_400529/best.pth


## 2. B-0 모델 복원
구형 체크포인트에 코드 버전이 없으면 기존 B0_init 또는 B30_init의 버전 기록을 참고한다. 이후 B-0 검증으로 복원을 확인한다.


In [2]:
b0_checkpoint = torch.load(b0_path, map_location="cpu", weights_only=True)
expected_commit = b0_checkpoint.get("dino_commit") or b0_checkpoint.get("config", {}).get("dino_commit")
version_source = "B-0 checkpoint"

# 구형 B-0 파일에 코드 버전이 없으면 기존 초기 파일의 기록을 참고한다.
if not expected_commit:
    initial_dir = Path("/content/drive/MyDrive/ondeviceAI/initial_checkpoints")
    for filename in ("B0_init.pth", "B30_init.pth"):
        candidate = initial_dir / filename
        if candidate.is_file():
            initial_metadata = torch.load(candidate, map_location="cpu", weights_only=True)
            expected_commit = initial_metadata.get("dino_commit")
            del initial_metadata
            if expected_commit:
                version_source = str(candidate)
                break

assert expected_commit, "DINOv3 코드 버전 기록이 없습니다. B0_init.pth 또는 B30_init.pth 경로를 확인하세요."
dino_dir = Path("/content/dinov3")
if not dino_dir.exists():
    subprocess.run(["git", "clone", "https://github.com/facebookresearch/dinov3.git", str(dino_dir)], check=True)
    subprocess.run(["git", "-C", str(dino_dir), "checkout", expected_commit], check=True)

current_commit = subprocess.check_output(
    ["git", "-C", str(dino_dir), "rev-parse", "HEAD"], text=True,
).strip()
assert current_commit == expected_commit, "기존 DINOv3 코드 버전이 다릅니다. 새 런타임에서 실행하세요."
if str(dino_dir) not in sys.path:
    sys.path.insert(0, str(dino_dir))
from dinov3.hub.backbones import dinov3_convnext_base

b0_model = dinov3_convnext_base(pretrained=False)
b0_model.head = nn.Linear(b0_model.embed_dim, 200)
print(b0_model.load_state_dict(b0_checkpoint["model_state_dict"], strict=True))
b0_saved_accuracy = b0_checkpoint["validation_accuracy"]
b0_metadata = {key: value for key, value in b0_checkpoint.items() if key != "model_state_dict"}
del b0_checkpoint
b0_model = b0_model.to(device)
print("코드 버전 참고 파일:", version_source)
print("DINOv3 코드 버전:", current_commit)
print("B-0 저장 당시 정확도:", f"{b0_saved_accuracy:.2%}")


<All keys matched successfully>
코드 버전 참고 파일: /content/drive/MyDrive/ondeviceAI/initial_checkpoints/B0_init.pth
DINOv3 코드 버전: 6876159a11b4df116f30f667f8c9888617df0751
B-0 저장 당시 정확도: 83.33%


## 3. 팀 데이터로더 준비


In [3]:
required_loader_files = [
    "data_loader.py",
    "docs/splits/train_ids.txt",
    "docs/splits/validation_ids.txt",
    "docs/splits/test_ids.txt",
    "docs/splits/metadata.json",
]

if not (repo_dir / ".git").is_dir():
    if repo_dir.exists() and any(repo_dir.iterdir()):
        raise RuntimeError(
            f"Git 저장소가 아닌 기존 폴더가 있습니다: {repo_dir}. "
            "기존 파일을 보존하고 repo_dir와 data_dir를 확인하세요."
        )
    subprocess.run(
        ["git", "clone", "--branch", DATA_REF, repo_url, str(repo_dir)],
        check=True,
    )
else:
    print("기존 팀 저장소를 사용합니다:", repo_dir)

cache_note = Path("/content/ondeviceAI_loader_cache.json")
working_files_ready = all((repo_dir / name).is_file() for name in required_loader_files)

if working_files_ready:
    loader_commit = subprocess.check_output(
        ["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True,
    ).strip()
    loader_origin = "existing_working_tree"
    loader_content = {
        name: (repo_dir / name).read_bytes() for name in required_loader_files
    }
elif cache_note.is_file():
    cached = json.loads(cache_note.read_text(encoding="utf-8"))
    cached_root = Path(cached["root"])
    if not all((cached_root / name).is_file() for name in required_loader_files):
        raise RuntimeError("데이터로더 캐시가 불완전합니다. 캐시 경로를 확인하세요.")
    loader_commit = cached["commit"]
    loader_origin = "existing_download_cache"
    loader_content = {
        name: (cached_root / name).read_bytes() for name in required_loader_files
    }
else:
    subprocess.run(
        ["git", "-C", str(repo_dir), "fetch", repo_url, DATA_REF], check=True,
    )
    loader_commit = subprocess.check_output(
        ["git", "-C", str(repo_dir), "rev-parse", "FETCH_HEAD"], text=True,
    ).strip()
    loader_origin = "downloaded_missing_files"
    loader_content = {
        name: subprocess.check_output(
            ["git", "-C", str(repo_dir), "show", f"{loader_commit}:{name}"]
        )
        for name in required_loader_files
    }

loader_file_hashes = {
    name: hashlib.sha256(content).hexdigest()
    for name, content in loader_content.items()
}
source_fingerprint = hashlib.sha256(
    json.dumps(loader_file_hashes, sort_keys=True).encode()
).hexdigest()

loader_snapshot = Path("/content/ondeviceAI_loader_runtime") / source_fingerprint[:16]
for name, content in loader_content.items():
    destination = loader_snapshot / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(content)

cache_note.write_text(
    json.dumps({"root": str(loader_snapshot), "commit": loader_commit}, indent=2),
    encoding="utf-8",
)

module_name = f"cub_data_{source_fingerprint[:16]}"
module_spec = importlib.util.spec_from_file_location(
    module_name, loader_snapshot / "data_loader.py",
)
cub = importlib.util.module_from_spec(module_spec)
sys.modules[module_name] = cub
module_spec.loader.exec_module(cub)

print("데이터로더 준비 방식:", loader_origin)
print("저장소 커밋:", loader_commit)
print("실제 파일 해시:", loader_file_hashes["data_loader.py"])


데이터로더 준비 방식: existing_working_tree
저장소 커밋: b8de137d50502d6d324f486db9c9d08e524ef707
실제 파일 해시: 792620d07b57bd5bed6f57fd84ddc3e844eb9ae9cad0c0728c16ab2f04749eb1


## 4. CUB 데이터 준비


In [4]:
metadata_files = [
    "images.txt", "image_class_labels.txt", "classes.txt", "train_test_split.txt",
]
data_ready = (
    (data_dir / "images").is_dir()
    and all((data_dir / name).is_file() for name in metadata_files)
)

if data_ready:
    print("기존 CUB 데이터를 사용합니다:", data_dir)
else:
    archive_path = Path("/content/CUB_200_2011.tgz")
    partial_path = Path("/content/CUB_200_2011.tgz.part")
    download_url = (
        "https://data.caltech.edu/records/65de6-vp158/"
        "files/CUB_200_2011.tgz?download=1"
    )

    if archive_path.is_file():
        downloaded_path = archive_path
        print("기존 CUB 압축파일을 사용합니다.")
    else:
        subprocess.run(
            ["wget", "-c", "-O", str(partial_path), download_url], check=True,
        )
        downloaded_path = partial_path

    with downloaded_path.open("rb") as file:
        checksum = hashlib.file_digest(file, "md5").hexdigest()
    assert checksum == "97eceeb196236b17998738112f37df78", (
        f"CUB 압축파일 검사 실패: {downloaded_path}. "
        "손상된 다운로드 파일을 정리한 뒤 다시 실행하세요."
    )

    if downloaded_path == partial_path:
        partial_path.replace(archive_path)

    data_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["tar", "-xzf", str(archive_path), "-C", str(data_dir.parent)],
        check=True,
    )

assert (data_dir / "images").is_dir()
assert all((data_dir / name).is_file() for name in metadata_files)
print("데이터 준비 완료:", data_dir)


데이터 준비 완료: /content/ondeviceAI/data/CUB_200_2011/CUB_200_2011


## 5. 공유 분할과 전처리 확인


In [5]:
image_list, image_labels, class_names = cub.read_cub_metadata(data_dir)
train_image_list, val_image_list, test_image_list = cub.split_cub_images(
    data_dir, image_list, image_labels,
)
train_transform, eval_transform = cub.build_transforms()

assert len(image_list) == 11788
assert set(image_labels.values()) == set(range(200))
assert set(class_names) == set(range(200))

source_metadata = json.loads(
    (loader_snapshot / "docs/splits/metadata.json").read_text(encoding="utf-8")
)
source_metadata_hashes = {}
for name in metadata_files:
    actual_hash = hashlib.sha256((data_dir / name).read_bytes()).hexdigest()
    source_metadata_hashes[name] = actual_hash
    expected_hash = source_metadata.get("source_sha256", {}).get(name)
    if expected_hash is not None:
        assert actual_hash == expected_hash, f"원본 메타데이터 불일치: {name}"

missing_images = [
    row.split(maxsplit=1)[1]
    for row in image_list
    if not (data_dir / "images" / row.split(maxsplit=1)[1]).is_file()
]
assert not missing_images, f"이미지 파일 누락: {len(missing_images)}장"

expected_counts = {"train": 4794, "validation": 1200, "test": 5794}
split_id_sets = {}
for split_name, rows in [
    ("train", train_image_list),
    ("validation", val_image_list),
    ("test", test_image_list),
]:
    actual_ids = [row.split()[0] for row in rows]
    reference_ids = (
        loader_snapshot / "docs/splits" / f"{split_name}_ids.txt"
    ).read_text().split()
    assert actual_ids == reference_ids, f"{split_name} 분할 순서 불일치"
    assert len(actual_ids) == expected_counts[split_name]
    assert len(set(actual_ids)) == len(actual_ids), f"{split_name} 중복 ID"
    split_id_sets[split_name] = set(actual_ids)
    print(f"{split_name}: {len(actual_ids)}장, ID와 순서 일치")

assert split_id_sets["train"].isdisjoint(split_id_sets["validation"])
assert split_id_sets["train"].isdisjoint(split_id_sets["test"])
assert split_id_sets["validation"].isdisjoint(split_id_sets["test"])

train_dataset = cub.BirdDataset(
    train_image_list, data_dir, image_labels, train_transform,
)
val_dataset = cub.BirdDataset(
    val_image_list, data_dir, image_labels, eval_transform,
)

print("학습 전처리:", train_transform)
print("검증 전처리:", eval_transform)


train: 4794장, ID와 순서 일치
validation: 1200장, ID와 순서 일치
test: 5794장, ID와 순서 일치
학습 전처리: Compose(
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.9, 1.1), contrast=None, saturation=None, hue=None)
    <function pad_image at 0x7e4cd4dd7380>
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)
검증 전처리: Compose(
    <function pad_image at 0x7e4cd4dd7380>
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


## 6. B-0 재평가
저장 당시 정확도와 검증 이미지 한 장 이내로 일치해야 pruning으로 진행한다. 테스트 평가는 하지 않는다.


In [6]:
assert b0_metadata["class_names"] == class_names, "B-0와 데이터로더의 클래스 순서가 다릅니다."
teacher_val_loader = DataLoader(
    val_dataset, batch_size=config["batch_size"], shuffle=False,
    num_workers=0, pin_memory=True,
)
criterion = nn.CrossEntropyLoss()


In [7]:
@torch.no_grad()
def evaluate_teacher(model, loader):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_images = 0

    for images, labels in tqdm(loader, desc="B-0 validation"):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # 기존 검증과 동일하게 FP32로 평가한다.
        with torch.autocast(device_type="cuda", enabled=False):
            logits = model(images)
            loss = criterion(logits, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_images += batch_size

    return total_loss / total_images, total_correct / total_images


b0_val_loss, b0_val_accuracy = evaluate_teacher(
    b0_model,
    teacher_val_loader,
)

print("저장 당시 정확도:", f"{b0_saved_accuracy:.2%}")
print("현재 검증 정확도:", f"{b0_val_accuracy:.2%}")
print("현재 검증 loss:", f"{b0_val_loss:.4f}")

# 환경 차이를 고려해 검증 이미지 1장 이내의 정확도 차이를 허용한다.
tolerance = 1 / len(teacher_val_loader.dataset) + 1e-12

assert abs(b0_val_accuracy - b0_saved_accuracy) <= tolerance, (
    "B-0 검증 결과가 이전과 다릅니다. "
    "Pruning 전에 데이터, 전처리와 모델 복원을 확인해야 합니다."
)

print("B-0 재평가 확인 완료")


B-0 validation:   0%|          | 0/150 [00:00<?, ?it/s]

저장 당시 정확도: 83.33%
현재 검증 정확도: 83.33%
현재 검증 loss: 0.7331
B-0 재평가 확인 완료


## 7. CPU 복사본 생성


In [8]:
import copy

# 원본 모델은 보관하고 CPU에서 복사본을 생성한다.
b0_model = b0_model.cpu()
b30_from_b0 = copy.deepcopy(b0_model)

params_before = sum(
    parameter.numel()
    for parameter in b30_from_b0.parameters()
)

head_before = b30_from_b0.head.weight.detach().clone()
head_bias_before = b30_from_b0.head.bias.detach().clone()

print("Pruning 전 parameter:", f"{params_before:,}")

Pruning 전 parameter: 87,771,464


## 8. 연결된 MLP 채널을 함께 제거하는 함수
각 블록의 축소 연산이 제거 대상 활성값을 0으로 만든 원래 연산과 일치하는지 검사한다.


In [9]:
@torch.no_grad()
def prune_trained_mlp(block, ratio=0.3):
    first = block.pwconv1
    second = block.pwconv2

    assert first.weight.device.type == "cpu"
    assert first.out_features == second.in_features

    hidden_before = first.out_features
    remove_count = round(hidden_before * ratio)
    keep_count = hidden_before - remove_count

    # 학습된 가중치의 출력 채널별 L1 중요도를 계산한다.
    importance = first.weight.abs().sum(dim=1)

    keep_indices = torch.argsort(
        importance,
        descending=True,
        stable=True,
    )[:keep_count].sort().values

    new_first = nn.Linear(first.in_features, keep_count)
    new_second = nn.Linear(keep_count, second.out_features)

    # 연결된 두 층에서 대응하는 채널을 함께 선택한다.
    new_first.weight.copy_(first.weight[keep_indices])
    new_first.bias.copy_(first.bias[keep_indices])
    new_second.weight.copy_(second.weight[:, keep_indices])
    new_second.bias.copy_(second.bias)

    # 원래 MLP에서 제거 대상의 활성값을 0으로 만든 결과와 비교한다.
    probe = torch.randn(2, first.in_features)
    mask = torch.zeros(hidden_before)
    mask[keep_indices] = 1

    reference = second(block.act(first(probe)) * mask)
    actual = new_second(block.act(new_first(probe)))

    torch.testing.assert_close(
        actual,
        reference,
        rtol=1e-4,
        atol=1e-4,
    )

    block.pwconv1 = new_first
    block.pwconv2 = new_second

    return {
        "hidden_before": hidden_before,
        "hidden_after": keep_count,
        "removed": remove_count,
        "keep_indices": keep_indices.tolist(),
    }

## 9. Pruning과 직후 검증
이후 회복 학습 전후를 비교할 기준을 기록한다.


In [10]:
assert all(
    block.pwconv1.out_features == 4 * block.pwconv1.in_features
    for stage in b30_from_b0.stages
    for block in stage
), "이미 pruning된 모델입니다. CPU 복사본 생성 셀부터 다시 실행하세요."

b30_from_b0_spec = {
    "source": "cub_finetuned_B0",
    "method": "per_block_pwconv1_weight_L1",
    "requested_ratio": 0.3,
    "rounding": "round(hidden_channels * ratio)",
    "blocks": {},
}

for stage_index, stage in enumerate(b30_from_b0.stages):
    for block_index, block in enumerate(stage):
        name = f"stages.{stage_index}.{block_index}"

        b30_from_b0_spec["blocks"][name] = prune_trained_mlp(
            block,
            ratio=0.3,
        )

# 학습된 분류층이 바뀌지 않았는지 확인한다.
assert torch.equal(b30_from_b0.head.weight, head_before)
assert torch.equal(b30_from_b0.head.bias, head_bias_before)

params_after = sum(
    parameter.numel()
    for parameter in b30_from_b0.parameters()
)

print("검사 완료 블록:", len(b30_from_b0_spec["blocks"]))
print("Pruning 후 parameter:", f"{params_after:,}")
print("전체 parameter 감소율:", f"{1 - params_after / params_before:.2%}")

b30_from_b0 = b30_from_b0.to(device)

pruned_loss, pruned_accuracy = evaluate_teacher(
    b30_from_b0,
    teacher_val_loader,
)

print("Pruning 전 검증 정확도:", f"{b0_val_accuracy:.2%}")
print("Pruning 직후 검증 정확도:", f"{pruned_accuracy:.2%}")
print("Pruning 직후 검증 loss:", f"{pruned_loss:.4f}")

검사 완료 블록: 36
Pruning 후 parameter: 62,633,144
전체 parameter 감소율: 28.64%


B-0 validation:   0%|          | 0/150 [00:00<?, ?it/s]

Pruning 전 검증 정확도: 83.33%
Pruning 직후 검증 정확도: 0.50%
Pruning 직후 검증 loss: 5.3586


## 10. 회복 학습 초기 파일 저장
이 파일은 CUB 학습을 마친 B-0에서 pruning한 상태다. 파일 저장 후 GPU 런타임을 다시 시작하고 두 번째 노트북을 실행한다.


In [11]:
recovery_init_dir = Path(
    "/content/drive/MyDrive/ondeviceAI/initial_checkpoints"
)
recovery_init_dir.mkdir(parents=True, exist_ok=True)

assert not recovery_init_path.exists(), "같은 이름의 초기 파일이 이미 있습니다. 기존 파일을 확인하세요."

recovery_checkpoint = {
    "format_version": 1,
    "checkpoint_type": "init",
    "model_name": "dinov3_convnext_base",
    "num_classes": 200,
    "dino_commit": current_commit,
    "source_stage": "cub_finetuned_B0",
    "source_checkpoint": str(b0_path),
    "source_epoch": b0_metadata["epoch"],
    "source_validation_accuracy": b0_saved_accuracy,
    "head_initialization": "preserved_from_finetuned_B0",
    "pruning_spec": b30_from_b0_spec,
    "class_names": class_names,
    "pruning_before_recovery_accuracy": pruned_accuracy,
    "pruning_before_recovery_loss": pruned_loss,
    "model_state_dict": {
        name: tensor.detach().cpu().clone()
        for name, tensor in b30_from_b0.state_dict().items()
    },
}

temporary_path = recovery_init_path.with_suffix(".tmp.pth")
torch.save(recovery_checkpoint, temporary_path)
temporary_path.replace(recovery_init_path)

del recovery_checkpoint

print("회복 학습 시작 파일:", recovery_init_path)

회복 학습 시작 파일: /content/drive/MyDrive/ondeviceAI/initial_checkpoints/B30_from_B0_init.pth
